# <center>🤖 Modèle de Prédiction du Défaut de Crédit</center>
<center><i>Pipeline scikit-learn — Classification binaire avec Optimisation avancée</i></center>

---

Ce notebook construit un **pipeline de machine learning complet** pour prédire le défaut de paiement, avec optimisation poussée des hyperparamètres.

**Plan :**
1. Chargement et nettoyage (reprise de l'EDA)
2. Feature engineering
3. Construction du Pipeline sklearn
4. Entraînement et comparaison de modèles
5. Évaluation détaillée du meilleur modèle
6. Optimisation avancée des hyperparamètres (2 phases)
   - Phase 1 : Recherche aléatoire large (RandomizedSearchCV)
   - Phase 2 : Affinage fin (GridSearchCV)
7. Analyse de l'importance des features
8. Sauvegarder le pipeline

## 1. Imports et chargement des données

In [1]:
# ── Librairies standard ─────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Scikit-learn : pipeline & préprocessing ─────────────────────────────────
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer

# ── Scikit-learn : modèles ──────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

# ── Scikit-learn : évaluation ───────────────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, ConfusionMatrixDisplay
)

print("✅ Imports OK")

✅ Imports OK


In [2]:
# ── Chargement du CSV ───────────────────────────────────────────────────────
df = pd.read_csv("credit_card_default.csv")
df = df.drop(columns=['predicted_default_payment_next_month'], errors='ignore')

print(f"Lignes : {df.shape[0]} | Colonnes : {df.shape[1]}")
df.head(3)

Lignes : 2965 | Colonnes : 25


,id,limit_balance,sex,education_level,marital_status,age,pay_0,pay_2,pay_3,pay_4,...,bill_amt_4,bill_amt_5,bill_amt_6,pay_amt_1,pay_amt_2,pay_amt_3,pay_amt_4,pay_amt_5,pay_amt_6,default_payment_next_month
0,27502.0,80000.0,1,6,1,54.0,0.0,0.0,0.0,0.0,...,29296.0,26210.0,17643.0,2545.0,2208.0,1336.0,2232.0,542.0,348.0,1
1,26879.0,200000.0,1,4,1,49.0,0.0,0.0,0.0,0.0,...,50146.0,50235.0,48984.0,1689.0,2164.0,2500.0,3480.0,2500.0,3000.0,0
2,18340.0,20000.0,2,6,2,22.0,0.0,0.0,0.0,0.0,...,1434.0,500.0,0.0,4641.0,1019.0,900.0,0.0,1500.0,0.0,1


## 2. Nettoyage (reprise de l'EDA)

In [3]:
# ── Conversion des colonnes de statut de paiement en float ──────────────────
pay_cols = ['pay_0', 'pay_2', 'pay_3', 'pay_4', 'pay_5', 'pay_6']
df[pay_cols] = df[pay_cols].astype(float)

# ── Suppression des modalités non documentées (voir EDA) ───────────────────
df = df[~df['education_level'].isin([0, 4, 5, 6])]
df = df[~df['marital_status'].isin([0, 3])]

print(f"Lignes après nettoyage : {df.shape[0]}")
print(f"Taux de défaut : {df['default_payment_next_month'].mean()*100:.1f}%")

Lignes après nettoyage : 2883
Taux de défaut : 21.7%


## 3. Feature Engineering

On crée des variables synthétiques pour enrichir le signal :
- **`max_delay`** : retard maximum observé sur les 6 mois (signal fort d'après l'EDA)
- **`mean_delay`** : retard moyen sur 6 mois
- **`total_bill`** : montant total facturé sur 6 mois
- **`total_paid`** : montant total remboursé sur 6 mois
- **`repay_ratio`** : ratio remboursement / facturation (capacité de remboursement)
- **`util_rate`** : taux d'utilisation du crédit (bill_amt_1 / limit_bal)

In [5]:
# ── Définition des listes de colonnes ──────────────────────────────────────
bill_cols  = ['bill_amt_1','bill_amt_2','bill_amt_3','bill_amt_4','bill_amt_5','bill_amt_6']  # Montants facturés (6 mois)
amt_cols   = ['pay_amt_1','pay_amt_2','pay_amt_3','pay_amt_4','pay_amt_5','pay_amt_6']      # Montants remboursés (6 mois)
delay_cols = ['pay_0','pay_2','pay_3','pay_4','pay_5','pay_6']                              # Statuts de retard de paiement

# ── Création des features synthétiques basées sur les retards ────────────────
df['max_delay']    = df[delay_cols].max(axis=1)        # Retard maximum observé sur 6 mois
df['mean_delay']   = df[delay_cols].mean(axis=1)       # Retard moyen sur 6 mois

# ── Création des features synthétiques basées sur les montants ───────────────
df['total_bill']   = df[bill_cols].sum(axis=1)         # Somme totale des factures (6 mois)
df['total_paid']   = df[amt_cols].sum(axis=1)          # Somme totale des remboursements (6 mois)

# ── Feature de capacité de remboursement ────────────────────────────────────
# Ratio remboursement / facturation : 
#   - ratio élevé = client paie bien (bon signal)
#   - ratio bas = client paie peu (mauvais signal)
df['repay_ratio']  = df['total_paid'] / (df['total_bill'].replace(0, np.nan))
df['repay_ratio']  = df['repay_ratio'].fillna(0).clip(0, 5)  # Remplir les 0/0 par 0 et limiter les valeurs extrêmes

# ── Feature de taux d'utilisation du crédit ────────────────────────────────
# Taux d'utilisation = facture la plus récente / limite de crédit
#   - ratio élevé = utilise beaucoup du crédit disponible
#   - ratio bas = utilise peu du crédit
df['util_rate']    = df['bill_amt_1'] / df['limit_balance'].replace(0, np.nan)
df['util_rate']    = df['util_rate'].fillna(0).clip(0, 5)  # Remplir les 0/0 par 0 et limiter les valeurs extrêmes

print("✅ Features créées :", ['max_delay','mean_delay','total_bill','total_paid','repay_ratio','util_rate'])

✅ Features créées : ['max_delay', 'mean_delay', 'total_bill', 'total_paid', 'repay_ratio', 'util_rate']


## 4. Définition des features et séparation Train / Test

In [6]:
TARGET = 'default_payment_next_month'

# ── Features numériques ─────────────────────────────────────────────────────
NUM_FEATURES = (
    ['limit_balance', 'age']
    + bill_cols
    + amt_cols
    + delay_cols
    + ['max_delay','mean_delay','total_bill','total_paid','repay_ratio','util_rate']
)

# ── Features catégorielles (encodage OHE) ───────────────────────────────────
CAT_FEATURES = ['sex', 'education_level', 'marital_status']

ALL_FEATURES = NUM_FEATURES + CAT_FEATURES

X = df[ALL_FEATURES]
y = df[TARGET]

# ── Split stratifié pour conserver le ratio de défaut ───────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train : {X_train.shape[0]} lignes | Test : {X_test.shape[0]} lignes")
print(f"Taux de défaut — Train : {y_train.mean()*100:.1f}% | Test : {y_test.mean()*100:.1f}%")

Train : 2306 lignes | Test : 577 lignes
Taux de défaut — Train : 21.7% | Test : 21.7%


### ✅ Vérification de la séparation Train/Test

La séparation stratifiée garantit que :
- ✓ Le ratio de défaut est conservé dans le train et le test
- ✓ Les données ne se chevauchent pas
- ✓ Le modèle sera évalué sur une distribution équilibrée

## 5. Construction du Pipeline sklearn

Le pipeline encapsule **toutes les transformations** et le modèle dans un seul objet,
ce qui garantit qu'aucune fuite de données entre train et test n'est possible.

In [11]:
# ── Transformateur numérique : imputation + normalisation ───────────────────
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),   # robuste aux outliers
    ('scaler',  StandardScaler()),                   # nécessaire pour LR et SVM
])

# ── Transformateur catégoriel : imputation + encodage OHE ──────────────────
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

# ── ColumnTransformer : applique chaque transformateur aux bonnes colonnes ──
preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, NUM_FEATURES),
    ('cat', cat_transformer, CAT_FEATURES),
])

print("✅ Préprocesseur défini")
print(f"   Colonnes numériques : {len(NUM_FEATURES)}")
print(f"   Colonnes catégorielles : {len(CAT_FEATURES)}")

✅ Préprocesseur défini
   Colonnes numériques : 26
   Colonnes catégorielles : 3


## 6. Comparaison de plusieurs modèles

On compare **3 modèles candidats** via **validation croisée stratifiée 5 folds** sur le score **ROC-AUC**
(plus adapté qu'accuracy face au déséquilibre de classes 79%/21%).

Les modèles testés :
- 🔵 Régression Logistique
- 🟢 Random Forest  
- 🟠 Gradient Boosting

In [15]:
# ── Dictionnaire des modèles candidats ─────────────────────────────────────
models = {
    'Régression Logistique': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest':         RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    'Gradient Boosting':     GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = {}

for name, model in models.items():
    # Assemblage du pipeline complet : préprocesseur + modèle
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier',   model),
    ])
    # Validation croisée 5 folds — métrique ROC-AUC
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='roc_auc', n_jobs=-1)
    results[name] = scores
    print(f"{name:30s} → ROC-AUC = {scores.mean():.4f} ± {scores.std():.4f}")

print("\n✅ Comparaison terminée")

Régression Logistique          → ROC-AUC = 0.7507 ± 0.0097
Random Forest                  → ROC-AUC = 0.7726 ± 0.0098
Gradient Boosting              → ROC-AUC = 0.7828 ± 0.0088

✅ Comparaison terminée

STRUCTURE DES PIPELINES PAR MODÈLE

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🔹 Pipeline : Régression Logistique
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['limit_balance', 'age',
                                                   'bil

In [ ]:
# ── Visualisation des scores de validation croisée ─────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

means = [v.mean() for v in results.values()]
stds  = [v.std()  for v in results.values()]
names = list(results.keys())

bars = ax.barh(names, means, xerr=stds, color=['#4C72B0','#55A868','#DD8452'],
               edgecolor='white', capsize=5)

for bar, m in zip(bars, means):
    ax.text(m + 0.002, bar.get_y() + bar.get_height()/2,
            f"{m:.4f}", va='center', fontweight='bold')

ax.set_xlabel('ROC-AUC (CV 5 folds)')
ax.set_title('Comparaison des modèles — Validation croisée', fontweight='bold')
ax.set_xlim(0.5, 1.0)
plt.tight_layout()
plt.show()

## 7. Évaluation du modèle baseline sur le Test Set

On retient le **Gradient Boosting** (généralement le plus performant sur ce type de données).
Ce modèle **baseline** sera ensuite optimisé via hyperparamètre tuning dans la section 8.
Adaptez le choix en fonction des résultats de la comparaison précédente.

In [ ]:
# ── Construction et entraînement du pipeline final ──────────────────────────
best_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   GradientBoostingClassifier(n_estimators=100, random_state=42)),
])

best_pipeline.fit(X_train, y_train)

# ── Prédictions ─────────────────────────────────────────────────────────────
y_pred      = best_pipeline.predict(X_test)
y_pred_proba = best_pipeline.predict_proba(X_test)[:, 1]

print("=== Rapport de classification ===")
print(classification_report(y_test, y_pred, target_names=['Pas de défaut', 'Défaut']))
print(f"ROC-AUC : {roc_auc_score(y_test, y_pred_proba):.4f}")

In [ ]:
# ── Matrice de confusion + Courbe ROC ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matrice de confusion
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Pas de défaut', 'Défaut'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title('Matrice de confusion', fontweight='bold')

# Courbe ROC
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
auc = roc_auc_score(y_test, y_pred_proba)
axes[1].plot(fpr, tpr, color='#4C72B0', lw=2, label=f'ROC-AUC = {auc:.4f}')
axes[1].plot([0,1],[0,1],'--', color='grey', label='Aléatoire')
axes[1].set_xlabel('Taux de faux positifs (FPR)')
axes[1].set_ylabel('Taux de vrais positifs (TPR)')
axes[1].set_title('Courbe ROC', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Optimisation avancée des hyperparamètres

**Approche en 2 phases** pour trouver les meilleurs hyperparamètres :

1. **Phase 1 — Recherche aléatoire large (RandomizedSearchCV)** :
   - Explore largement l'espace des paramètres (50 itérations)
   - Identifie rapidement les zones prometteuses
   - Paramètres testés : `n_estimators`, `max_depth`, `learning_rate`, `subsample`, `min_samples_split`, `min_samples_leaf`, `max_features`

2. **Phase 2 — Affinage fin (GridSearchCV)** :
   - Affine les meilleurs paramètres trouvés en Phase 1
   - Validation croisée stratifiée 5 folds
   - Recherche systématique dans une grille réduite autour des optimums
   
Cette approche hybride est **plus efficace qu'une grille seule** en termes de temps de calcul et de qualité des résultats.

In [ ]:
# ── Imports pour l'optimisation avancée ────────────────────────────────────
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from scipy.stats import randint, uniform
import numpy as np

# ── ÉTAPE 1 : Recherche aléatoire large avec RandomizedSearchCV ──────────────
print("=" * 80)
print("PHASE 1: Recherche aléatoire large (RandomizedSearchCV)")
print("=" * 80)

param_dist = {
    'classifier__n_estimators':   randint(50, 300),
    'classifier__max_depth':      randint(2, 15),
    'classifier__min_samples_split': randint(2, 20),
    'classifier__min_samples_leaf':  randint(1, 10),
    'classifier__learning_rate':  uniform(0.001, 0.3),
    'classifier__subsample':      uniform(0.5, 0.5),
    'classifier__max_features':   ['sqrt', 'log2'],
}

random_search = RandomizedSearchCV(
    estimator   = best_pipeline,
    param_distributions = param_dist,
    n_iter      = 50,              # 50 itérations aléatoires
    cv          = StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring     = 'roc_auc',
    n_jobs      = -1,
    verbose     = 2,
    random_state = 42,
)

random_search.fit(X_train, y_train)

print(f"\n✅ Meilleurs paramètres (Random Search) : {random_search.best_params_}")
print(f"   Meilleur ROC-AUC (CV) : {random_search.best_score_:.4f}")

# ── ÉTAPE 2 : Affinage autour des meilleurs paramètres (GridSearchCV) ────────
print("\n" + "=" * 80)
print("PHASE 2: Affinage fin (GridSearchCV) autour des meilleurs paramètres")
print("=" * 80)

# Extraire les meilleurs paramètres et affiner autour
best_params = random_search.best_params_
param_grid_fine = {
    'classifier__n_estimators':   [max(50, best_params['classifier__n_estimators'] - 30),
                                   best_params['classifier__n_estimators'],
                                   best_params['classifier__n_estimators'] + 30],
    'classifier__max_depth':      [max(2, best_params['classifier__max_depth'] - 2),
                                   best_params['classifier__max_depth'],
                                   best_params['classifier__max_depth'] + 2],
    'classifier__learning_rate':  [max(0.001, best_params['classifier__learning_rate'] - 0.05),
                                   best_params['classifier__learning_rate'],
                                   min(0.3, best_params['classifier__learning_rate'] + 0.05)],
    'classifier__subsample':      [max(0.5, best_params['classifier__subsample'] - 0.1),
                                   best_params['classifier__subsample'],
                                   min(1.0, best_params['classifier__subsample'] + 0.1)],
}

grid_search = GridSearchCV(
    estimator   = best_pipeline,
    param_grid  = param_grid_fine,
    cv          = StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring     = 'roc_auc',
    n_jobs      = -1,
    verbose     = 1,
)

grid_search.fit(X_train, y_train)

print(f"\n✅ Meilleurs paramètres (Grid Search affiné) : {grid_search.best_params_}")
print(f"   Meilleur ROC-AUC (CV) : {grid_search.best_score_:.4f}")

# ── Comparaison des résultats ────────────────────────────────────────────────
print("\n" + "=" * 80)
print("RÉSUMÉ DES OPTIMISATIONS")
print("=" * 80)
print(f"ROC-AUC Random Search  : {random_search.best_score_:.4f}")
print(f"ROC-AUC Grid Search    : {grid_search.best_score_:.4f}")
print(f"\n✨ Meilleur modèle sélectionné : {max(random_search.best_score_, grid_search.best_score_):.4f}")


In [ ]:
# ── Sélection du meilleur modèle entre Random Search et Grid Search ────────
best_model = (grid_search if grid_search.best_score_ >= random_search.best_score_ 
              else random_search).best_estimator_

y_pred_opt       = best_model.predict(X_test)
y_pred_proba_opt = best_model.predict_proba(X_test)[:, 1]

print("=" * 80)
print("RÉSULTATS FINAUX — MODÈLE OPTIMISÉ")
print("=" * 80)
print("\nRapport de classification :")
print(classification_report(y_test, y_pred_opt, target_names=['Pas de défaut', 'Défaut']))
print(f"ROC-AUC Test Set : {roc_auc_score(y_test, y_pred_proba_opt):.4f}")

# ── Comparaison avant/après optimisation ────────────────────────────────────
print("\n" + "-" * 80)
print("COMPARAISON AVANT/APRÈS OPTIMISATION")
print("-" * 80)
print(f"Score avant optimisation (baseline) : {roc_auc_score(y_test, best_pipeline.predict_proba(X_test)[:, 1]):.4f}")
print(f"Score après optimisation           : {roc_auc_score(y_test, y_pred_proba_opt):.4f}")


In [ ]:
# ── Visualisation des résultats de recherche ───────────────────────────────
import pandas as pd

# Créer un DataFrame avec les résultats
results_df = pd.DataFrame(random_search.cv_results_)

# Top 10 des meilleurs résultats
top_10 = results_df.nlargest(10, 'mean_test_score')[['rank_test_score', 'mean_test_score', 'std_test_score']]

print("\nTop 10 des meilleures itérations (Random Search) :")
print(top_10.to_string())

# ── Graphique : Évolution des scores ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Scores en fonction du numéro d'itération
axes[0].plot(results_df['mean_test_score'], marker='o', linestyle='-', alpha=0.6)
axes[0].axhline(y=grid_search.best_score_, color='r', linestyle='--', 
                label=f'Meilleur (Grid Search): {grid_search.best_score_:.4f}')
axes[0].set_xlabel('Itération')
axes[0].set_ylabel('ROC-AUC Score (CV)')
axes[0].set_title('Progression des scores (Random Search)', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Distribution des scores
axes[1].hist(results_df['mean_test_score'], bins=20, color='#4C72B0', edgecolor='black', alpha=0.7)
axes[1].axvline(x=random_search.best_score_, color='r', linestyle='--', linewidth=2,
                label=f'Meilleur: {random_search.best_score_:.4f}')
axes[1].set_xlabel('ROC-AUC Score')
axes[1].set_ylabel('Fréquence')
axes[1].set_title('Distribution des scores (50 itérations)', fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


## 9. Analyse de l'importance des features

Extrait les poids d'importance du modèle Gradient Boosting optimisé pour identifier les features les plus déterminantes dans la prédiction du défaut.

In [ ]:
# ── Récupération des noms de features après transformation OHE ──────────────
ohe_feature_names = (
    best_model.named_steps['preprocessor']
    .named_transformers_['cat']
    .named_steps['ohe']
    .get_feature_names_out(CAT_FEATURES)
    .tolist()
)
all_feature_names = NUM_FEATURES + ohe_feature_names

# ── Importances du modèle final ─────────────────────────────────────────────
importances = best_model.named_steps['classifier'].feature_importances_

feat_imp = pd.Series(importances, index=all_feature_names).sort_values(ascending=False)

# ── Visualisation : Top 20 features ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
feat_imp.head(20).plot(kind='barh', ax=ax, color='#4C72B0', edgecolor='white')
ax.invert_yaxis()
ax.set_title('Top 20 — Importance des features (Gradient Boosting)', fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

print("\nTop 10 features :")
print(feat_imp.head(10).to_string())

## 10. Déploiement — Sauvegarder le pipeline optimisé

Le pipeline final **optimisé** (préprocesseur + modèle avec hyperparamètres affinés) est sérialisé avec `joblib` pour une utilisation en production ou sur de nouvelles données.

In [ ]:
import joblib

# Sauvegarde du pipeline complet (préprocesseur + modèle)
joblib.dump(best_model, 'credit_default_pipeline.pkl')
print("✅ Pipeline sauvegardé : credit_default_pipeline.pkl")

# Chargement et vérification
loaded_pipeline = joblib.load('credit_default_pipeline.pkl')
test_pred = loaded_pipeline.predict_proba(X_test.head(5))[:, 1]
print(f"Vérification — 5 premières probabilités de défaut : {test_pred.round(3)}")

---

## 🎯 Récapitulatif

| Étape | Choix technique | Justification |
|---|---|---|
| **Préprocessing** | `StandardScaler` + `OneHotEncoder` dans un `ColumnTransformer` | Zéro fuite train/test, tout dans le pipeline |
| **Feature engineering** | `max_delay`, `repay_ratio`, `util_rate` | Insights de l'EDA : retard = signal fort |
| **Modèle** | Gradient Boosting | Robuste aux outliers, capture les non-linéarités |
| **Évaluation** | ROC-AUC + CV 5 folds stratifiée | Adapté au déséquilibre 79/21 |
| **Optimisation Phase 1** | `RandomizedSearchCV` (50 itérations) | Exploration large et rapide de l'espace |
| **Optimisation Phase 2** | `GridSearchCV` (affinage fin) | Recherche systématique autour des optimums |
| **Sélection finale** | Meilleur modèle entre les 2 phases | Garantit le meilleur score possible |
| **Déploiement** | `joblib` | Pipeline entier sérialisé, prêt pour la production |

### 📊 Résultats clés
- **Modèle baseline** : Gradient Boosting sans optimisation
- **Modèle optimisé** : Gradient Boosting avec hyperparamètres affinés (2 phases)
- **Amélioration** : Comparaison avant/après visible dans la cellule d'évaluation
- **Features les plus importantes** : Identifiées via `feature_importances_`